# Parallel Execution — Fan-Out, Race, MapReduce, and Batch

---

This notebook covers Multigen's parallel execution primitives:

- **`Parallel`** — run N agents concurrently on the same context
- **`FanOut`** — distribute a list of items across one agent
- **`MapReduce`** — parallel map phase followed by a sequential reduce
- **`Race`** — first result wins, losers are cancelled
- **`Batch`** — process a large list in fixed-size chunks

All examples are self-contained and runnable without API keys.

## 1. Why Parallel? The Latency Collapse Argument

When agents are independent — they don't depend on each other's outputs — running them
sequentially wastes time. Consider three agents with latencies of 100 ms, 200 ms, 150 ms:

```
Sequential execution:
──────────────────────────────────────────────────────────────────────────
agent_A [100ms]──── agent_B [200ms]──────────────── agent_C [150ms]────────
────────────────────────────────────────────────────────────────────────────
0ms                100ms                           300ms                  450ms

Total latency = 100 + 200 + 150 = 450 ms

Parallel execution:
─────────────────────────────────────────────────────
agent_A [100ms]────
agent_B [200ms]────────────────
agent_C [150ms]───────────
─────────────────────────────────────────────────────
0ms                                               200ms

Total latency = max(100, 200, 150) = 200 ms
Speedup: 450ms → 200ms = 2.25x faster
```

With 10 agents of 200ms each: sequential = 2000ms, parallel = 200ms.
This is why parallel execution is critical for any LLM workflow that makes multiple
independent API calls (e.g., fetching data from 5 sources, running 4 classifiers in parallel).

In [ ]:
import asyncio
import time
import random

from multigen import (
    Parallel,
    FanOut,
    MapReduce,
    Race,
    Batch,
    FunctionAgent,
    Chain,
    agent,
)

print("Multigen parallel primitives imported.")

## 2. Basic Parallel — 3 Agents Running Concurrently

`Parallel` takes a list of agents and runs them all at the same time using `asyncio.gather`.
Each agent receives the same input context. Their outputs are collected into a
`ParallelResult` object.

```python
Parallel(
    agents=[agent_a, agent_b, agent_c],
    name="my_parallel",
)
```

After running:
- `result.outputs` — list of dicts, one per agent, in the same order as `agents`
- `result.all_succeeded` — True if every agent completed without error
- `result.merged` — all outputs merged into a single dict (last writer wins on key conflicts)
- `result.total_ms` — wall-clock time (should be close to `max(individual_latencies)`)

In [ ]:
# Three independent analysis agents with different simulated latencies
@agent
async def fetch_price(ctx: dict) -> dict:
    """Fetch asset price from exchange A (fast, 50ms)."""
    await asyncio.sleep(0.05)
    return {"price": 42_500.00, "source": "exchange_A"}

@agent
async def fetch_volume(ctx: dict) -> dict:
    """Fetch 24h volume from exchange B (medium, 120ms)."""
    await asyncio.sleep(0.12)
    return {"volume_24h": 18_200_000, "source_vol": "exchange_B"}

@agent
async def fetch_sentiment(ctx: dict) -> dict:
    """Fetch social sentiment score (slow, 180ms)."""
    await asyncio.sleep(0.18)
    return {"sentiment_score": 0.73, "sentiment_source": "social_feed"}

# Build the parallel executor
market_parallel = Parallel(
    agents=[fetch_price, fetch_volume, fetch_sentiment],
    name="market_data_fetch",
)

t0 = time.perf_counter()
result = await market_parallel.run({"asset": "BTC"})
elapsed = (time.perf_counter() - t0) * 1000

print(f"Wall time : {elapsed:.0f} ms  (expected ~180ms, not 50+120+180=350ms)")
print("all_succeeded:", result.all_succeeded)
print()
print("outputs[0] (fetch_price)   :", result.outputs[0])
print("outputs[1] (fetch_volume)  :", result.outputs[1])
print("outputs[2] (fetch_sentiment):", result.outputs[2])
print()
print("merged (all keys combined) :", result.merged)

## 3. `concurrency=` Parameter — Throttling Parallel Execution

When you have many agents to run in parallel, running all of them simultaneously can
overwhelm downstream services (rate limits, DB connection pools, etc.).

The `concurrency=N` parameter caps the maximum number of agents running at once using
an `asyncio.Semaphore`. Agents are started in batches of N; when one finishes, the next
queued agent starts.

```
10 agents, concurrency=3:

Wave 1: [agent_0, agent_1, agent_2]  ← running
Queue : [agent_3, agent_4, ..., agent_9]  ← waiting

As each finishes → agent_3 starts, then agent_4, etc.
```

In [ ]:
# Create 10 agents that each take ~50ms
def make_worker(worker_id: int) -> FunctionAgent:
    async def worker(ctx: dict) -> dict:
        t_start = time.perf_counter()
        await asyncio.sleep(0.05)  # 50ms per worker
        t_end = time.perf_counter()
        return {f"worker_{worker_id}": f"done in {(t_end - t_start)*1000:.0f}ms"}
    return FunctionAgent(fn=worker, name=f"worker_{worker_id}")

workers = [make_worker(i) for i in range(10)]

# Unconstrained: all 10 run at once (~50ms total)
p_unconstrained = Parallel(agents=workers, name="unconstrained")

t0 = time.perf_counter()
_ = await p_unconstrained.run({})
print(f"Unconstrained (10 workers, no limit): {(time.perf_counter()-t0)*1000:.0f}ms expected ~50ms")

# Throttled: at most 3 run at once (~4 waves × 50ms = ~200ms)
p_throttled = Parallel(agents=workers, concurrency=3, name="throttled")

t0 = time.perf_counter()
_ = await p_throttled.run({})
print(f"Throttled (10 workers, concurrency=3): {(time.perf_counter()-t0)*1000:.0f}ms expected ~200ms")

## 4. `fail_fast=True` — Abort on First Error

With `fail_fast=True` (the default), if **any** agent raises an exception, the `Parallel`
immediately cancels all remaining agents and re-raises the exception.

Use this when you cannot proceed with partial data — e.g., all three sub-tasks are
required to produce a valid output.

In [ ]:
@agent
async def good_agent_1(ctx: dict) -> dict:
    await asyncio.sleep(0.02)
    return {"g1": "ok"}

@agent
async def bad_agent(ctx: dict) -> dict:
    await asyncio.sleep(0.01)  # fails quickly
    raise RuntimeError("Database connection lost")

@agent
async def good_agent_2(ctx: dict) -> dict:
    await asyncio.sleep(0.05)  # would succeed if given time
    return {"g2": "ok"}

# fail_fast=True: one error aborts everything
p_strict = Parallel(
    agents=[good_agent_1, bad_agent, good_agent_2],
    fail_fast=True,
    name="strict_parallel",
)

try:
    r = await p_strict.run({})
except RuntimeError as e:
    print(f"Parallel raised RuntimeError: {e}")
    print("good_agent_2 was cancelled before it could finish.")

## 5. `fail_fast=False` — Collect All Results, Even Partial

With `fail_fast=False`, all agents run to completion (or failure). The `ParallelResult`
contains outputs from successful agents and exceptions from failed ones.

This is the right choice for **best-effort aggregation** — e.g., fetch data from 5 sources
and use whatever comes back, even if 2 of the 5 fail.

In [ ]:
# Same agents, but this time we tolerate the failure
p_resilient = Parallel(
    agents=[good_agent_1, bad_agent, good_agent_2],
    fail_fast=False,
    name="resilient_parallel",
)

r = await p_resilient.run({})

print("all_succeeded :", r.all_succeeded)  # False
print("outputs       :", r.outputs)         # [{'g1': 'ok'}, None, {'g2': 'ok'}]
print("errors        :")
for agent_name, exc in r.errors.items():
    print(f"  {agent_name}: {type(exc).__name__}: {exc}")

# Filter out None values and use what succeeded
successful_outputs = [o for o in r.outputs if o is not None]
print("Successful outputs:", successful_outputs)

## 6. Timeout — `Parallel(..., timeout=10.0)`

The `timeout=` parameter applies to the **entire parallel group** — if not all agents
complete within the timeout, the parallel executor cancels any still-running agents
and returns (or raises, depending on `fail_fast`) immediately.

This gives you a hard SLA on the parallel step as a whole.

In [ ]:
@agent
async def fast_source(ctx: dict) -> dict:
    await asyncio.sleep(0.05)  # 50ms
    return {"fast": "data"}

@agent
async def very_slow_source(ctx: dict) -> dict:
    await asyncio.sleep(5.0)   # 5 seconds — will be cancelled
    return {"slow": "data"}

# With fail_fast=False and a 200ms timeout:
# fast_source finishes, very_slow_source is cancelled after 200ms
p_timeout = Parallel(
    agents=[fast_source, very_slow_source],
    timeout=0.2,        # 200ms wall-clock limit for the whole group
    fail_fast=False,    # collect partial results
    name="timeout_demo",
)

t0 = time.perf_counter()
r = await p_timeout.run({})
elapsed = (time.perf_counter() - t0) * 1000

print(f"Wall time: {elapsed:.0f}ms  (stopped by timeout at ~200ms, not 5000ms)")
print("all_succeeded:", r.all_succeeded)   # False
print("outputs      :", r.outputs)          # [{'fast': 'data'}, None]
print("errors       :", {k: type(v).__name__ for k, v in r.errors.items()})

## 7. FanOut — Distributing a List Across One Agent

`FanOut` takes a **single agent** and a **list of items** and runs the agent once for
each item, in parallel. Each invocation receives the item as a separate context key.

This is the Multigen equivalent of `asyncio.gather(*[agent.run({item_key: item}) for item in items])`.

```python
FanOut(
    agent=my_agent,
    input_key="item",     # context key each invocation receives
    items_key="items",    # which context key holds the list
    output_key="results", # where to store the list of outputs
)
```

In [ ]:
# Single agent that processes one URL at a time
@agent
async def scrape_url(ctx: dict) -> dict:
    """Simulate scraping a URL (returns mock metadata)."""
    url = ctx.get("item", "")
    await asyncio.sleep(0.05)  # simulate HTTP request
    return {
        "url": url,
        "title": f"Page title for {url.split('/')[-1]}",
        "word_count": random.randint(200, 2000),
    }

# FanOut distributes the list across the agent
scraper_fanout = FanOut(
    agent=scrape_url,
    input_key="item",      # each agent invocation gets ctx['item'] = one URL
    items_key="urls",      # the list lives in ctx['urls']
    output_key="scraped",  # results are written to ctx['scraped']
    name="url_scraper_fanout",
)

urls = [
    "https://example.com/page1",
    "https://example.com/page2",
    "https://example.com/page3",
    "https://example.com/page4",
    "https://example.com/page5",
]

t0 = time.perf_counter()
result = await scraper_fanout.run({"urls": urls})
elapsed = (time.perf_counter() - t0) * 1000

print(f"Scraped {len(urls)} URLs in {elapsed:.0f}ms  (5 × 50ms in parallel = ~50ms)")
print()
for item in result["scraped"]:
    print(f"  {item['url'].split('/')[-1]:6s}: {item['title']}  ({item['word_count']} words)")

## 8. FanOut with `reduce_fn` — Custom Aggregation After Fan-Out

After all fan-out invocations complete, a `reduce_fn` can aggregate their outputs into
a single value. This is useful when you want to summarise or sort the results.

```python
FanOut(
    agent=...,
    reduce_fn=lambda results: sorted(results, key=lambda r: r['score'], reverse=True),
)
```

In [ ]:
# Agent that scores a document
@agent
async def score_document(ctx: dict) -> dict:
    """Assign a relevance score to a document."""
    doc = ctx.get("item", {})
    await asyncio.sleep(0.02)
    # Mock scoring: longer docs score higher, plus random noise
    score = len(doc.get("text", "")) * 0.01 + random.uniform(0, 0.5)
    return {"doc_id": doc["id"], "text": doc["text"], "score": round(score, 3)}

# reduce_fn: sort by score descending, return top 3
def top3_reducer(results: list) -> list:
    """Sort results by score and return the top 3."""
    sorted_results = sorted(results, key=lambda r: r["score"], reverse=True)
    return sorted_results[:3]

ranking_fanout = FanOut(
    agent=score_document,
    input_key="item",
    items_key="documents",
    output_key="ranked_docs",
    reduce_fn=top3_reducer,
    name="document_ranker",
)

documents = [
    {"id": f"doc_{i}", "text": "word " * (i * 10)}
    for i in range(1, 9)  # 8 documents of varying lengths
]

result = await ranking_fanout.run({"documents": documents})
print("Top 3 ranked documents:")
for rank, doc in enumerate(result["ranked_docs"], 1):
    print(f"  #{rank}: {doc['doc_id']}  score={doc['score']}")

## 9. MapReduce — Parallel Map + Sequential Reduce

`MapReduce` formalises the classic pattern:

1. **Map phase**: run a `map_agent` on every item in a list, in parallel
2. **Reduce phase**: call `reduce_agent` once with all mapped outputs as a list

This is perfect for workflows like:
- Extract keywords from 50 documents in parallel, then union all keywords
- Score 100 candidates in parallel, then sort and return top N
- Translate 20 paragraphs in parallel, then concatenate

```python
MapReduce(
    map_agent=per_item_agent,
    reduce_agent=aggregator_agent,
    items_key="documents",
    output_key="final_result",
)
```

In [ ]:
# Map agent: extract keywords from a single document
@agent
async def extract_keywords(ctx: dict) -> dict:
    """Extract top 5 non-stopword keywords from a single document."""
    await asyncio.sleep(0.03)  # simulate NLP processing
    doc = ctx.get("item", "")
    stopwords = {"the", "a", "is", "in", "on", "at", "of", "and", "to", "it", "this"}
    words = [w.strip(".,").lower() for w in doc.split()]
    keywords = list({w for w in words if w not in stopwords and len(w) > 3})
    return {"keywords": keywords[:5]}

# Reduce agent: union all keyword lists into a single frequency-ranked list
@agent
async def union_keywords(ctx: dict) -> dict:
    """Combine keyword lists from all documents, rank by frequency."""
    all_mapped = ctx.get("mapped_outputs", [])
    freq = {}
    for item_output in all_mapped:
        for kw in item_output.get("keywords", []):
            freq[kw] = freq.get(kw, 0) + 1
    ranked = sorted(freq.items(), key=lambda x: x[1], reverse=True)
    return {"global_keywords": [kw for kw, _ in ranked[:10]], "keyword_freq": dict(ranked[:10])}

keyword_mapreduce = MapReduce(
    map_agent=extract_keywords,
    reduce_agent=union_keywords,
    items_key="documents",
    mapped_key="mapped_outputs",  # key passed to reduce agent
    output_key="analysis",
    name="keyword_mapreduce",
)

documents = [
    "Machine learning models require large training datasets and compute resources.",
    "Deep learning neural networks have transformed computer vision tasks significantly.",
    "Natural language processing enables machines to understand human language.",
    "Reinforcement learning trains agents through rewards and penalties in environments.",
    "Transfer learning allows models trained on large datasets to adapt to new tasks.",
]

t0 = time.perf_counter()
result = await keyword_mapreduce.run({"documents": documents})
elapsed = (time.perf_counter() - t0) * 1000

print(f"MapReduce over {len(documents)} docs in {elapsed:.0f}ms")
print("Global keywords:", result["analysis"]["global_keywords"])
print("Keyword frequencies:", result["analysis"]["keyword_freq"])

## 10. Race — First Result Wins, Others Cancelled

`Race` runs multiple agents simultaneously and returns as soon as the **first one completes
successfully**. All other still-running agents are cancelled.

Use cases:
- **Redundant data sources**: query 3 sources and use whichever responds first
- **A/B testing**: run two model variants and use the faster response
- **Hedged requests**: send the same request to 2 replicas for lower tail latency

The `Race` result contains:
- `result.winner` — name of the agent that won
- `result.output` — that agent's output dict
- `result.total_ms` — time from start to first completion

In [ ]:
# Three database replicas — we query all three and use whichever responds first
@agent
async def replica_east(ctx: dict) -> dict:
    """US-East replica — usually 150ms but can spike."""
    latency = 0.15 + random.uniform(0, 0.3)  # 150-450ms with noise
    await asyncio.sleep(latency)
    return {"data": "row_42_v1", "replica": "east", "latency_ms": latency * 1000}

@agent
async def replica_west(ctx: dict) -> dict:
    """US-West replica — usually 80ms but can spike."""
    latency = 0.08 + random.uniform(0, 0.3)  # 80-380ms with noise
    await asyncio.sleep(latency)
    return {"data": "row_42_v1", "replica": "west", "latency_ms": latency * 1000}

@agent
async def replica_eu(ctx: dict) -> dict:
    """EU replica — usually 200ms but can spike."""
    latency = 0.20 + random.uniform(0, 0.3)  # 200-500ms with noise
    await asyncio.sleep(latency)
    return {"data": "row_42_v1", "replica": "eu", "latency_ms": latency * 1000}

# Race: whichever replica responds first wins
hedged_query = Race(
    agents=[replica_east, replica_west, replica_eu],
    name="hedged_db_query",
)

# Run 3 times to see different winners
for trial in range(3):
    result = await hedged_query.run({"key": "row_42"})
    print(f"Trial {trial+1}: winner={result.winner:15s}  "
          f"latency={result.output['latency_ms']:.0f}ms  "
          f"data={result.output['data']}")

## 11. Batch — Processing Large Lists in Fixed-Size Chunks

`Batch` splits a large input list into fixed-size chunks and processes each chunk
as a `Parallel` group. This gives you controlled parallelism across a large workload
without running all items at once (which could overwhelm rate limits).

```
25 items, batch_size=5:

Chunk 1 → [item_0..item_4]  → run in parallel → collect results
Chunk 2 → [item_5..item_9]  → run in parallel → collect results
Chunk 3 → [item_10..item_14] → run in parallel → collect results
Chunk 4 → [item_15..item_19] → run in parallel → collect results
Chunk 5 → [item_20..item_24] → run in parallel → collect results
                                ↓
                          All 25 results
```

Total latency = `ceil(N / batch_size) × max_item_latency` — much better than sequential
but bounded so you don't overwhelm services.

In [ ]:
# Agent that processes one order at a time
@agent
async def process_order(ctx: dict) -> dict:
    """Validate and enrich a single order record."""
    order = ctx.get("item", {})
    await asyncio.sleep(0.04)  # 40ms per order (simulated DB write)
    return {
        "order_id": order["id"],
        "total": order["qty"] * order["price"],
        "status": "processed",
    }

# Generate 25 mock orders
orders = [
    {"id": f"ORD-{i:04d}", "qty": random.randint(1, 10), "price": round(random.uniform(5, 500), 2)}
    for i in range(25)
]

# Batch: process 25 orders in chunks of 5
order_batch = Batch(
    agent=process_order,
    items_key="orders",
    item_key="item",
    output_key="processed_orders",
    batch_size=5,
    name="order_processor_batch",
)

t0 = time.perf_counter()
result = await order_batch.run({"orders": orders})
elapsed = (time.perf_counter() - t0) * 1000

processed = result["processed_orders"]
print(f"Processed {len(processed)} orders in {elapsed:.0f}ms")
print(f"  Expected: ceil(25/5) × 40ms = 5 × 40ms = ~200ms")
print()
print("First 5 results:")
for o in processed[:5]:
    print(f"  {o['order_id']}: total=${o['total']:.2f}  status={o['status']}")

## 12. Combining Patterns — FanOut Inside a Chain

All parallel primitives produce agents with the same `run(ctx) -> result` interface,
so they can be used as steps inside a `Chain`, nested inside another `FanOut`, or
combined in any other way.

Here we build a chain where:
1. A `tokenise` step splits a document into sentences
2. A `FanOut` runs a sentiment scorer on each sentence in parallel
3. A `summarise_sentiments` step aggregates the per-sentence scores

In [ ]:
# Step 1: split document into sentences
@agent
async def split_sentences(ctx: dict) -> dict:
    """Split ctx['text'] into a list of sentences."""
    text = ctx.get("text", "")
    sentences = [s.strip() for s in text.split(".") if s.strip()]
    return {"sentences": sentences}

# Step 2 (inner agent for FanOut): score one sentence
@agent
async def score_sentence(ctx: dict) -> dict:
    """Mock sentiment score for one sentence."""
    await asyncio.sleep(0.02)
    sentence = ctx.get("item", "")
    positive_words = {"great", "good", "excellent", "happy", "love", "best"}
    negative_words = {"bad", "terrible", "hate", "worst", "poor", "awful"}
    words = set(sentence.lower().split())
    score = sum(1 for w in words if w in positive_words) - sum(1 for w in words if w in negative_words)
    return {"sentence": sentence, "score": score}

# Step 2 (FanOut wrapper): run score_sentence on every sentence in parallel
sentence_fanout = FanOut(
    agent=score_sentence,
    input_key="item",
    items_key="sentences",
    output_key="sentence_scores",
    name="per_sentence_sentiment",
)

# Step 3: aggregate sentence scores
@agent
async def aggregate_sentiments(ctx: dict) -> dict:
    """Compute overall sentiment from per-sentence scores."""
    scores = [item["score"] for item in ctx.get("sentence_scores", [])]
    if not scores:
        return {"overall_sentiment": "neutral", "avg_score": 0.0}
    avg = sum(scores) / len(scores)
    label = "positive" if avg > 0 else "negative" if avg < 0 else "neutral"
    return {"overall_sentiment": label, "avg_score": round(avg, 3)}

# Chain: split → fanout → aggregate
sentiment_pipeline = Chain(
    steps=[split_sentences, sentence_fanout, aggregate_sentiments],
    name="full_sentiment_pipeline",
)

text = (
    "The product quality is great and I love it. "
    "Delivery was terrible and packaging was poor. "
    "Customer service was excellent and very helpful."
)

result = await sentiment_pipeline.run({"text": text})

print("Per-sentence scores:")
for item in result.full_context["sentence_scores"]:
    print(f"  score={item['score']:+d}  '{item['sentence'][:60]}'")

print()
print("Overall sentiment:", result.final_output["overall_sentiment"])
print("Average score    :", result.final_output["avg_score"])

## 13. Real-World Example: Parallel Financial Data Fetch from 4 Sources

A realistic scenario: fetch market data from 4 independent financial APIs in parallel,
then aggregate into a single market snapshot.

This demonstrates:
- `Parallel` with `fail_fast=False` for resilient multi-source fetching
- A custom merge step inside a `Chain`
- Handling partial failures gracefully

In [ ]:
# Four financial data source agents
@agent
async def fetch_bloomberg(ctx: dict) -> dict:
    """Bloomberg terminal feed (premium, reliable, 80ms)."""
    await asyncio.sleep(0.08)
    ticker = ctx.get("ticker", "AAPL")
    return {"source": "bloomberg", "price": 185.42, "bid": 185.40, "ask": 185.44, "volume": 12_400_000}

@agent
async def fetch_refinitiv(ctx: dict) -> dict:
    """Refinitiv Eikon feed (100ms)."""
    await asyncio.sleep(0.10)
    return {"source": "refinitiv", "price": 185.43, "pe_ratio": 28.7, "eps": 6.46}

@agent
async def fetch_iex(ctx: dict) -> dict:
    """IEX Cloud (free tier, occasionally fails, 60ms)."""
    await asyncio.sleep(0.06)
    # Simulate 20% failure rate on IEX
    if random.random() < 0.2:
        raise ConnectionError("IEX Cloud rate limit exceeded")
    return {"source": "iex", "price": 185.41, "market_cap": 2_890_000_000_000}

@agent
async def fetch_yahoo(ctx: dict) -> dict:
    """Yahoo Finance scraper (slow, 200ms, free)."""
    await asyncio.sleep(0.20)
    return {"source": "yahoo", "price": 185.44, "52w_high": 198.23, "52w_low": 155.98}

# Parallel fetch from all 4 sources simultaneously
data_fetch = Parallel(
    agents=[fetch_bloomberg, fetch_refinitiv, fetch_iex, fetch_yahoo],
    fail_fast=False,   # some sources may fail — use what we get
    timeout=0.5,       # hard cap: don't wait more than 500ms total
    name="market_data_parallel",
)

# Aggregation step that runs after the parallel fetch
@agent
async def build_snapshot(ctx: dict) -> dict:
    """Merge outputs from all successful sources into a market snapshot."""
    raw_outputs = ctx.get("parallel_outputs", [])
    # Filter out None (failed sources)
    successful = [o for o in raw_outputs if o is not None]
    # Compute VWAP-style price from available prices
    prices = [o["price"] for o in successful if "price" in o]
    avg_price = sum(prices) / len(prices) if prices else None
    # Flatten all metadata fields
    snapshot = {"sources_used": [o["source"] for o in successful], "consensus_price": avg_price}
    for o in successful:
        snapshot.update({k: v for k, v in o.items() if k != "source"})
    return {"market_snapshot": snapshot}

# Full pipeline: parallel fetch → aggregate
# Note: Parallel results need to be injected into ctx before the next step.
# We use a FunctionAgent adapter to bridge Parallel output into the context.
async def run_market_pipeline(ticker: str) -> dict:
    ctx = {"ticker": ticker}
    
    # Run parallel fetch
    fetch_result = await data_fetch.run(ctx)
    ctx["parallel_outputs"] = fetch_result.outputs
    ctx["fetch_errors"] = {k: str(v) for k, v in fetch_result.errors.items()}
    
    # Run aggregation
    snapshot_delta = await build_snapshot.run(ctx)
    ctx.update(snapshot_delta)
    
    return ctx

# Run the pipeline
t0 = time.perf_counter()
final_ctx = await run_market_pipeline("AAPL")
elapsed = (time.perf_counter() - t0) * 1000

snapshot = final_ctx["market_snapshot"]
print(f"Market data pipeline completed in {elapsed:.0f}ms")
print(f"Sources used    : {snapshot['sources_used']}")
print(f"Consensus price : ${snapshot['consensus_price']:.2f}")
if final_ctx.get("fetch_errors"):
    print(f"Failed sources  : {list(final_ctx['fetch_errors'].keys())}")
print()
print("Full snapshot:")
for k, v in snapshot.items():
    if k not in ("sources_used",):
        print(f"  {k:20s}: {v}")

## Summary

| Primitive | Use When | Key Parameters |
|---|---|---|
| `Parallel` | Run N independent agents on the same context | `agents`, `concurrency`, `fail_fast`, `timeout` |
| `FanOut` | Run one agent on each item in a list | `agent`, `items_key`, `input_key`, `reduce_fn` |
| `MapReduce` | Parallel map + sequential reduce | `map_agent`, `reduce_agent`, `items_key` |
| `Race` | First result wins, rest cancelled | `agents` |
| `Batch` | Large list in controlled-size chunks | `agent`, `batch_size`, `items_key` |

All primitives plug into `Chain` as ordinary steps — there is no special treatment needed.
The uniform `run(ctx) -> result` interface is what makes this composition seamless.

**Next notebook**: `15_graph_dag_local.ipynb` — dependency-aware parallel execution with
directed acyclic graphs, conditional edges, and dynamic routing.